### NLBSE'24 SetFit Baseline

# Data Rearranging Script

In [ ]:
from datasets import load_dataset, concatenate_datasets

ds = load_dataset(
    "csv",
    data_files={
        "train": "data/issues_train.csv",
        "test": "data/issues_test.csv",
    }
)

combined = concatenate_datasets([ds["train"], ds["test"]])

In [ ]:
def map_valid_invalid(example):
    return {
        "label": 1 if example["label"] == "bug" else 0
    }

In [ ]:
from datasets import ClassLabel

combined = combined.map(map_valid_invalid)
combined = combined.cast_column("label", ClassLabel(names=['invalid', 'valid']))

Casting the dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
new_ds = combined.train_test_split(
    train_size=2500,
    test_size=500,
    stratify_by_column="label",
    seed=42,
)

In [ ]:
new_ds

DatasetDict({
    train: Dataset({
        features: ['repo', 'created_at', 'label', 'title', 'body'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['repo', 'created_at', 'label', 'title', 'body'],
        num_rows: 500
    })
})

In [ ]:
new_ds["train"].to_csv("data/new_train.csv", index=False)
new_ds["test"].to_csv("data/new_test.csv", index=False)

print("Train and test datasets saved to data/new_train.csv and data/new_test.csv")

Creating CSV from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Train and test datasets saved to data/new_train.csv and data/new_test.csv


# Original Script

In [1]:
!pip install -q setfit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00


In [2]:
BASE_MODEL = "sentence-transformers/all-mpnet-base-v2"
RANDOM_SEED = 42
OUTPUT_PATH = 'output/setfit'

!mkdir -p $OUTPUT_PATH

In [3]:
DATA_PATH = 'data'

!mkdir -p $DATA_PATH

In [15]:
from datasets import Dataset, DatasetDict

main_ds = Dataset.from_csv("data/sampled_issues.csv")

test_ds = main_ds.filter(lambda example: example["test_tag"] == 1)
train_ds = main_ds.filter(lambda example: example["test_tag"] == 0)

ds = DatasetDict({ "train": train_ds, "test": test_ds })
ds

DatasetDict({
    train: Dataset({
        features: ['repository_url', 'issue_number', 'title_processed', 'body_processed', 'label', 'label_cat', 'test_tag', 'valid'],
        num_rows: 8051
    })
    test: Dataset({
        features: ['repository_url', 'issue_number', 'title_processed', 'body_processed', 'label', 'label_cat', 'test_tag', 'valid'],
        num_rows: 1949
    })
})

In [16]:
import re

def process_dataset(example):

    # concatenate title and body
    text = (example['title_processed'] or "") + " " + (example['body_processed'] or "")

    example['text'] = text
    return example

In [17]:
ds = ds.shuffle(seed=RANDOM_SEED)
ds = ds.map(process_dataset)
ds = ds.select_columns(['issue_number', 'valid', 'text'])
ds = ds.rename_column('valid', 'label')
ds

DatasetDict({
    train: Dataset({
        features: ['issue_number', 'label', 'text'],
        num_rows: 8051
    })
    test: Dataset({
        features: ['issue_number', 'label', 'text'],
        num_rows: 1949
    })
})

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
BASE_SAVE_DIR = "/content/drive/MyDrive/CS559/project/mpnet_models"

In [18]:
import wandb
from datetime import datetime
from setfit import SetFitModel, Trainer, TrainingArguments
import os

group = datetime.utcnow().replace(microsecond=0).isoformat()

references = {}
predictions = {}

wandb.init(
    project="NLBSE'24 Issue Report Classification - SetFit",
    group=group,
    name="bug_validity_classification",
)

model = SetFitModel.from_pretrained(BASE_MODEL)

args = TrainingArguments(
    output_dir=OUTPUT_PATH,
    save_strategy="no",
    report_to="wandb",
    logging_steps=1,
    seed=RANDOM_SEED,
    batch_size=(64, 2),
    num_epochs=1,
    num_iterations=5,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
)

trainer.train()

/tmp/ipython-input-1451849375.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  group = datetime.utcnow().replace(microsecond=0).isoformat()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
/usr/local/lib/python3.12/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
/usr/local/lib/python3.12/

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tevfikemresungur (tevfikemresungur-bilkent-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:279: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:279: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())


...

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())


Map:   0%|          | 0/8051 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 80510
  Batch size = 64
  Num epochs = 1
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)


Step,Training Loss
1,0.383200
2,0.332000
3,0.384700
4,0.403200
5,0.413800
6,0.394900
7,0.338300
8,0.402000
9,0.392600
10,0.397600


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
save_path = os.path.join(BASE_SAVE_DIR, "best_model")

model.save_pretrained(save_path)

In [20]:
test_set = ds["test"]

references = list(test_set['label'])

predictions = list(model.predict(test_set['text'], batch_size=8, show_progress_bar=True))

wandb.finish()

Batches:   0%|          | 0/244 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


train/embedding_loss,█▆▆▅▅▄▄▄▄▄▄▄▃▃▃▃▂▂▃▃▂▂▃▂▄▃▁▂▂▁▂▂▂▁▂▁▁▁▁▂
train/epoch,▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▆▆▆▇▇▇▇▇▇████
train/grad_norm,▃▁▁▁▁▂▃▂▂▂▂▂▂▃▃▄▂▄▃▃▃▃▄▄▄█▃▃▂▄▂▄▃▃▃▄▅▂▁█
train/learning_rate,▃▄███▇▇▇▇▆▆▆▆▆▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
total_flos,0
train/embedding_loss,0.0756
train/epoch,1
train/global_step,1258
train/grad_norm,1.68707
train/learning_rate,0.0


In [21]:
from sklearn.metrics import classification_report
from numpy import mean

metrics = ['precision', 'recall', 'f1-score']

results = classification_report(references, predictions, digits=4, output_dict=True)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [22]:
results

{'invalid': {'precision': 0.8358608385370205,
  'recall': 0.865988909426987,
  'f1-score': 0.8506581933726737,
  'support': 1082.0},
 'valid': {'precision': 0.8248792270531401,
  'recall': 0.7877739331026529,
  'f1-score': 0.8058997050147493,
  'support': 867.0},
 'accuracy': 0.8311954848640328,
 'macro avg': {'precision': 0.8303700327950803,
  'recall': 0.8268814212648199,
  'f1-score': 0.8282789491937115,
  'support': 1949.0},
 'weighted avg': {'precision': 0.830975739944653,
  'recall': 0.8311954848640328,
  'f1-score': 0.8307476703319757,
  'support': 1949.0}}

In [23]:
import json
import os

output_file_name = 'results.json'

labels = ['invalid','valid']

with open(os.path.join(OUTPUT_PATH, output_file_name), 'w') as fp:
    json.dump(results, fp, indent=2)

print(f"Repository{' '*15}Label     Precision  Recall     F1")
for repo in ['overall']:
  print("-"*63)
  for label in labels:
    out = f"{repo:<25}{label:<10}"
    for metric in metrics:
      out += f"{results[label][metric]:<10.4f} "
    print(out)

Repository               Label     Precision  Recall     F1
---------------------------------------------------------------
overall                  invalid   0.8359     0.8660     0.8507     
overall                  valid     0.8249     0.7878     0.8059     
